# Coding-docs RAG poisoning - Week 1 (data + index)

Run the **setup cell first in every new session**, then only the cell for today.
Every step is a script in `coding_track/steps/`; this notebook just calls them.
All outputs go to `MyDrive/coding_track/` (see `coding_track/README.md`).

| Day | Runtime to pick (Runtime > Change runtime type) | Cells |
|---|---|---|
| 1 | High-RAM CPU, setup with `NEED_OLLAMA = True` | Day 1 |
| 2-5 | **High-RAM CPU** | Day 2 ... Day 5 |
| 6 | **L4 GPU** | Day 6 (and Day 7 in the same session if time allows) |
| 7 | L4 (same session as Day 6) or High-RAM CPU | Day 7 |

Steps are safe to re-run: finished downloads, shards and files are detected and skipped.

In [ ]:
# ===== SETUP: run first in every session =====
REPO_URL = "https://github.com/<your-username>/<your-repo>.git"   # <-- set this once
REPO_DIR = "/content/repo"
NEED_OLLAMA = True          # True on Day 1 (and from Week 2 on)
OLLAMA_MODELS = ["llama3.2:3b"]

import os, subprocess, time
from google.colab import drive

def sh(cmd):
    print("$", cmd)
    subprocess.run(cmd, shell=True, check=True)

drive.mount("/content/drive")
if os.path.isdir(REPO_DIR):
    sh(f"git -C {REPO_DIR} pull --ff-only")
else:
    sh(f"git clone {REPO_URL} {REPO_DIR}")
os.chdir(REPO_DIR)
sh("pip install -q -r coding_track/requirements-colab.txt")

if NEED_OLLAMA:
    sh("apt-get -qq install -y pciutils zstd > /dev/null")   # pciutils helps Ollama find a GPU
    sh("curl -fsSL https://ollama.com/install.sh | sh")
    env = dict(os.environ, OLLAMA_CONTEXT_LENGTH="16384")
    subprocess.Popen(["ollama", "serve"], env=env,
                     stdout=open("/content/ollama.log", "w"), stderr=subprocess.STDOUT)
    time.sleep(5)
    for m in OLLAMA_MODELS:
        sh(f"ollama pull {m}")
    sh("ollama ps; ollama list")
print("setup done")

## Day 1 (Mon) - Ollama context check  
Needs the setup cell with `NEED_OLLAMA = True`. Expect `PASS` and `prompt_eval_count` around 9,500.

In [ ]:
!python -m coding_track.steps.day1_ollama_check

## Day 2 (Tue) - Download everything (High-RAM CPU)  
Prints each dataset's real columns and row count. Stack Overflow is ~2M posts.

In [ ]:
!python -m coding_track.steps.day2_download

## Day 3 (Wed) - Tag tasks, pick rare libraries, split, check fake names on PyPI

In [ ]:
!python -m coding_track.steps.day3_tasks_splits_targets

## Day 4 (Thu) - Fresh docs, then the Stack Overflow filter and overlap list

In [ ]:
!python -m coding_track.steps.day4_fresh_docs

In [ ]:
!python -m coding_track.steps.day4_filter_stackoverflow

## Day 5 (Fri) - Clean, de-duplicate, manifest, then chunk

In [ ]:
!python -m coding_track.steps.day5_clean_and_manifest

In [ ]:
!python -m coding_track.steps.day5_chunk

## Day 6 (Sat) - L4 GPU: embed in shards, then build ChromaDB on /content and zip it to Drive  
If the session drops, run the setup cell and this cell again: finished shards are skipped.

In [ ]:
!nvidia-smi -L
!python -m coding_track.steps.day6_embed

In [ ]:
!python -m coding_track.steps.day6_build_chroma

## Day 7 (Sun) - Count check, 20 eyeballed questions, NDCG@10, notes file

In [ ]:
!python -m coding_track.steps.day7_check_index

## Any time - regenerate the notes file from the saved stats

In [ ]:
!python -m coding_track.steps.week1_notes